# مشروع استخراج وتصحيح نصوص الخط اليدوي

### **الخلية 1: تثبيت المكتبات المطلوبة**

In [3]:
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara
!pip install -q easyocr paddleocr pytesseract transformers torch torchvision fpdf2 Levenshtein
!pip install -q peft huggingface_hub datasets opencv-python-headless langdetect langchain-text-splitters

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-ara is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


### **الخلية 2: الاستيرادات وإعداد المسارات**

In [4]:
import os, cv2, torch, sqlite3, io, numpy as np, pandas as pd, json, re, base64
from PIL import Image
from google.colab import drive, userdata
from datetime import datetime

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/Handwriting_Dataset'
DB_PATH = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ Hugging Face Token loaded")
except:
    print("⚠️ No HF_TOKEN found")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

Mounted at /content/drive
✅ Hugging Face Token loaded
🚀 Device: cuda


### **الخلية 3: تحميل النماذج (TrOCR، EasyOCR، المدققات)**

In [ ]:
import pytesseract
from paddleocr import PaddleOCR
import easyocr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("⏳ Loading OCR models...")

paddle_ocr = None
try:
    paddle_ocr = PaddleOCR(use_angle_cls=True, lang='ar', show_log=False)
    print("✅ PaddleOCR loaded")
except Exception as e:
    print(f"⚠️ PaddleOCR failed: {e}")

easy_reader = None
try:
    easy_reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
    print("✅ EasyOCR loaded")
except Exception as e:
    print(f"⚠️ EasyOCR failed: {e}")

trocr_proc, trocr_model = None, None
try:
    trocr_proc = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)
    print("✅ TrOCR loaded")
except Exception as e:
    print(f"⚠️ TrOCR failed: {e}")

def enhance_digit_recognition(text):
    if not text: return text
    corrections = {'O':'0', 'o':'0', 'I':'1', 'l':'1', '|':'1', 'Z':'2', 'z':'2',
                   'S':'5', 's':'5', 'G':'6', 'g':'6', 'T':'7', 't':'7', 'B':'8', 'b':'8'}
    for wrong, correct in corrections.items():
        text = text.replace(wrong, correct)
    return text

def get_ensemble_predictions(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    results = []

    if trocr_model and trocr_proc:
        try:
            pixel_values = trocr_proc(images=pil_img, return_tensors="pt").pixel_values.to(device)
            out = trocr_model.generate(pixel_values)
            results.append(trocr_proc.batch_decode(out, skip_special_tokens=True)[0])
        except: pass

    try:
        results.append(pytesseract.image_to_string(pil_img, config='--psm 7').strip())
    except: pass

    if paddle_ocr:
        try:
            p_res = paddle_ocr.ocr(img_bgr, cls=False)
            if p_res and p_res[0]:
                results.append(" ".join([l[1][0] for l in p_res[0]]))
        except: pass

    if easy_reader:
        try:
            e_res = easy_reader.readtext(img_bgr)
            results.append(" ".join([it[1] for it in e_res]))
        except: pass

    unique = list(dict.fromkeys([enhance_digit_recognition(r) for r in results if r and r.strip()]))
    return unique[:3]

/tmp/ipykernel_772/3893972768.py:10: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  paddle_ocr = PaddleOCR(use_angle_cls=True, lang='ar', show_log=False)


⏳ Loading OCR models...
⚠️ PaddleOCR failed: Unknown argument: show_log
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ EasyOCR loaded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ TrOCR loaded


In [ ]:
# تحويل نماذج EasyOCR لتكون دائمة على Drive
import os

drive_easyocr_path = '/content/drive/MyDrive/Handwriting_Dataset/.EasyOCR'
local_easyocr_path = os.path.expanduser('~/.EasyOCR')

# إذا لم تكن النماذج منقولة مسبقاً إلى Drive، قم بنقلها
if not os.path.exists(drive_easyocr_path) and os.path.exists(local_easyocr_path):
    print('جاري نقل نماذج EasyOCR إلى Drive للمرة الأولى...')
    !mv {local_easyocr_path} {drive_easyocr_path}

# إنشاء الرابط الرمزي لضمان وصول المكتبة للنماذج من Drive
if not os.path.islink(local_easyocr_path):
    if os.path.exists(local_easyocr_path):
        !rm -rf {local_easyocr_path}
    !ln -s {drive_easyocr_path} {local_easyocr_path}
    print('✅ تم ربط نماذج EasyOCR بـ Google Drive بنجاح')

✅ تم ربط نماذج EasyOCR بـ Google Drive بنجاح


### **الخلية 4: دوال المعالجة المسبقة والتعرف والتصحيح**

In [ ]:
from huggingface_hub import HfApi, login

def extract_bilingual_vocab():
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='verified' ORDER BY page_num, y, x", conn)
    if words.empty:
        print("⚠️ لا توجد بيانات موثقة")
        return None
    vocab_pairs = []
    for page in words['page_num'].unique():
        p_words = words[words['page_num'] == page]
        texts = p_words['predicted_text'].astype(str).tolist()
        en = [t for t in texts if t and all(ord(c) < 128 for c in t)]
        ar = [t for t in texts if t and any('\u0600' <= c <= '\u06FF' for c in t)]
        if en and ar:
            vocab_pairs.append({'english': ' | '.join(en[:3]), 'arabic': ' | '.join(ar[:3]), 'page': page})
    df = pd.DataFrame(vocab_pairs)
    out_path = os.path.join(OUTPUT_DIR, 'bilingual_vocab.csv')
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"✅ تم استخراج القاموس إلى {out_path}")
    return df

def upload_to_hf(repo_name="handwriting-ocr-dataset"):
    if not os.environ.get('HF_TOKEN'):
        print("❌ لا يوجد HF_TOKEN")
        return
    login(token=os.environ['HF_TOKEN'])
    jsonl_path = os.path.join(OUTPUT_DIR, "data.jsonl")
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("SELECT image_data, predicted_text FROM handwriting_data WHERE status='verified'", conn)
    if len(df) == 0:
        print("⚠️ لا توجد بيانات مصادقة لتصديرها")
        return
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            img_b64 = base64.b64encode(row['image_data']).decode('utf-8')
            f.write(json.dumps({"image": img_b64, "text": row['predicted_text']}, ensure_ascii=False) + '\n')
    api = HfApi()
    username = userdata.get('HF_USERNAME', 'dr_abdulmalek')
    repo_id = f"{username}/{repo_name}"
    try:
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    except: pass
    api.upload_file(path_or_fileobj=jsonl_path, path_in_repo="data.jsonl", repo_id=repo_id, repo_type="dataset")
    print(f"🚀 تم الرفع إلى https://huggingface.co/datasets/{repo_id}")

### **الخلية 5: دالة معالجة PDF وحفظ البيانات في SQLite مع تسجيل الإحصائيات**

In [ ]:
def process_pdf(pages_start=1, pages_end=2, resume=True):
    proc_start = time.time()
    checkpoint = load_checkpoint() if resume else None
    if checkpoint:
        print(f"🔄 استئناف من الصفحة {checkpoint['last_page_processed']}")
        pages_start = checkpoint['last_page_processed']

    correction_dict = build_correction_dict()
    try:
        images = convert_from_path(PDF_PATH, dpi=300, first_page=pages_start, last_page=pages_end)
    except Exception as e:
        logging.error(f"فشل معالجة PDF: {e}"); return

    total_words_processed = checkpoint.get('processed_words', 0) if checkpoint else 0
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
                        (image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                         image_data BLOB, predicted_text TEXT, status TEXT,
                         confidence REAL, model_source TEXT,
                         x INTEGER, y INTEGER, w INTEGER, h INTEGER, page_num INTEGER)''')

        for idx, pil in enumerate(images):
            actual_page = pages_start + idx
            img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
            try: detections = easy_reader.readtext(img_bgr, detail=1)
            except: detections = []

            binary = preprocess_image(img_bgr)
            boxes_info = []
            if detections:
                for (bbox, text, conf) in detections:
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    boxes_info.append(((x,y,w,h), (bbox, text, conf)))
            else:
                manual_boxes = smart_word_segmentation(img_bgr, binary)
                for box in manual_boxes: boxes_info.append((box, None))

            for (box, e_raw) in boxes_info:
                x, y, w, h = box
                crop = img_bgr[y:y+h, x:x+w]
                raw, conf, src, _ = recognize_word_ensemble(crop, easyocr_raw=e_raw)
                final = apply_correction_dict(correct_text(raw), correction_dict)
                _, buf = cv2.imencode(".png", crop)
                conn.execute('''INSERT INTO handwriting_data
                                (image_data, predicted_text, status, confidence, model_source, x, y, w, h, page_num)
                                VALUES (?,?,?,?,?,?,?,?,?,?)''',
                             (buf.tobytes(), final, 'unverified', conf, src, x, y, w, h, actual_page))
                total_words_processed += 1

            save_checkpoint(actual_page + 1, pages_end, total_words_processed)
            patches.cv2_imshow(cv2.resize(img_bgr, (0,0), fx=0.3, fy=0.3))
        conn.commit()

    clear_checkpoint()
    stats = {"timestamp": datetime.now().isoformat(), "pages": pages_end-pages_start+1, "words": total_words_processed}
    with open(STATS_JSON, 'w', encoding='utf-8') as f: json.dump(stats, f, ensure_ascii=False)
    logging.info(f"✅ اكتملت المعالجة في {time.time()-proc_start:.2f} ثانية.")

### **الخلية 6: تشغيل عملية الاستخراج (يُرجى تعديل عدد الصفحات حسب الحاجة)**

In [ ]:
# اختر نطاق الصفحات (مثال: من 1 إلى 2)
process_pdf(pages_start=1, pages_end=2, resume=True)

NameError: name 'time' is not defined

### **الخلية 7: الواجهة التفاعلية المتكاملة (v2)**

In [ ]:
# يُرجى تشغيل الخلية التالية أولاً لتعريف الدوال، ثم استدعاء الواجهة من هناك.

### **الخلية 8: عرض مسارات ملفات المراقبة (للإشارة)**

In [ ]:
print("\n📁 ملفات المراقبة والتطوير:")
print(f"   • سجل الأحداث: {LOG_FILE}")
print(f"   • إحصائيات المعالجة: {STATS_JSON}")
print(f"   • تصحيحات المستخدم: {FEEDBACK_CSV}")

In [ ]:
# =============== الخلية 8: تصدير البيانات الموثقة فقط ===============
def export_finetuning_dataset(output_dir="hf_training_dataset", val_ratio=0.1):
    import random
    os.makedirs(output_dir, exist_ok=True)
    img_dir = os.path.join(output_dir, "images")
    os.makedirs(img_dir, exist_ok=True)

    with sqlite3.connect(DB_PATH) as conn:
        # تصدير الكلمات الموثقة سواء من واجهة الكلمات أو واجهة الجمل
        df_verified = pd.read_sql_query('''
            SELECT image_id, image_data, predicted_text, status
            FROM handwriting_data
            WHERE status IN ('verified', 'sentence_corrected')
            ORDER BY image_id
        ''', conn)

    if df_verified.empty:
        print("⚠️ لا توجد بيانات موثقة (Verified). قم بمراجعة بعض الكلمات وتأكيدها أولاً.")
        return None

    jsonl_records = []
    for _, row in df_verified.iterrows():
        filename = f"img_{row['image_id']}.png"
        filepath = os.path.join(img_dir, filename)
        with open(filepath, "wb") as f:
            f.write(row['image_data'])
        jsonl_records.append({
            "image": filename,
            "text": row['predicted_text'].strip() if row['predicted_text'] else "",
            "verified": True
        })

    random.shuffle(jsonl_records)
    split_idx = int(len(jsonl_records) * (1 - val_ratio))
    train_data = jsonl_records[:split_idx]
    val_data = jsonl_records[split_idx:]

    def save_jsonl(data, fname):
        path = os.path.join(output_dir, fname)
        with open(path, "w", encoding="utf-8") as f:
            for rec in data:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        return path

    save_jsonl(train_data, "train.jsonl")
    save_jsonl(val_data, "val.jsonl")

    print(f"✅ تم التصدير بنجاح: {len(jsonl_records)} عينة موثقة.")
    print(f"📁 المسار: {os.path.abspath(output_dir)}")
    return output_dir

In [ ]:
def reconstruct_sentences(y_tolerance=25):
    import pandas as pd
    import sqlite3
    from langdetect import detect

    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='verified' ORDER BY page_num, y, x", conn)

    if words.empty: return

    all_sentences = []
    for page in words['page_num'].unique():
        p_words = words[words['page_num'] == page].copy()
        lines = []
        if not p_words.empty:
            curr_line = [p_words.iloc[0].to_dict()]
            for i in range(1, len(p_words)):
                row = p_words.iloc[i].to_dict()
                if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                    curr_line.append(row)
                else:
                    lines.append(curr_line); curr_line = [row]
            lines.append(curr_line)

        for line in lines:
            text_preview = ' '.join([str(w['predicted_text']) for w in line])
            try: lang = detect(text_preview)
            except: lang = 'en'

            # إصلاح الترتيب: RTL للعربي (x تنازلي)
            sorted_line = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            sentence = ' '.join([str(w['predicted_text']) for w in sorted_line])
            all_sentences.append({'page': page, 'text': sentence, 'lang': lang})

    df_res = pd.DataFrame(all_sentences)
    display(df_res)
    return df_res

In [ ]:
from huggingface_hub import HfApi, login
from peft import get_peft_model, LoraConfig, TaskType
from torch.optim import AdamW
from torch.utils.data import Dataset as TorchDataset, DataLoader

def push_to_huggingface(hf_repo_id, hf_token, local_dataset_dir="hf_training_dataset"):
    if not os.path.exists(local_dataset_dir):
        return print("⚠️ شغّل export_finetuning_dataset() أولاً.")
    login(token=hf_token)
    api = HfApi()
    try: api.create_repo(repo_id=hf_repo_id, repo_type="dataset", exist_ok=True)
    except Exception as e: print(f"Repo info: {e}")
    api.upload_folder(folder_path=local_dataset_dir, repo_id=hf_repo_id, repo_type="dataset")
    print(f"✅ تم الرفع: https://huggingface.co/datasets/{hf_repo_id}")

def finetune_trocr_lora(min_samples=100):
    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query("SELECT image_data, predicted_text FROM handwriting_data WHERE status IN ('verified', 'sentence_corrected')", conn)
    if len(df_v) < min_samples: return print(f"⚠️ لديك {len(df_v)} عينة فقط. انتظر حتى {min_samples}.")
    print(f"🚀 بدء التدريب على {len(df_v)} عينة...")
    lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, target_modules=["query", "value"], lora_dropout=0.1)
    model = get_peft_model(trocr_model, lora_config)
    model.train()
    class HandwritingDataset(TorchDataset):
        def __init__(self, df): self.df = df
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            pixel_values = trocr_processor(images=img, return_tensors="pt").pixel_values.squeeze()
            labels = trocr_processor.tokenizer(row['predicted_text'], return_tensors="pt", padding="max_length", max_length=50).input_ids.squeeze()
            labels[labels == trocr_processor.tokenizer.pad_token_id] = -100
            return {"pixel_values": pixel_values, "labels": labels}
    loader = DataLoader(HandwritingDataset(df_v), batch_size=4, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=5e-5)
    for epoch in range(3):
        total_loss = 0
        for batch in loader:
            out = model(pixel_values=batch["pixel_values"].to(device), labels=batch["labels"].to(device))
            out.loss.backward(); optimizer.step(); optimizer.zero_grad()
            total_loss += out.loss.item()
        print(f"Epoch {epoch+1}/3 | Loss: {total_loss/len(loader):.4f}")
    lora_save_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
    model.save_pretrained(lora_save_path)
    trocr_processor.save_pretrained(lora_save_path)
    global trocr_model
    trocr_model = model
    print(f"✅ تم الحفظ والتشغيل من: {lora_save_path}")

In [ ]:
def edit_correction_dict_ui():
    import json, ipywidgets as widgets
    from IPython.display import display, clear_output
    dict_path = os.path.join(OUTPUT_DIR, 'correction_dict.json')
    data = {}
    if os.path.exists(dict_path):
        with open(dict_path, 'r', encoding='utf-8') as f: data = json.load(f)

    out = widgets.Output()
    key_in = widgets.Text(description="أصلي:")
    val_in = widgets.Text(description="تصحيح:")
    def save(b):
        if key_in.value:
            data[key_in.value] = val_in.value
            with open(dict_path, 'w', encoding='utf-8') as f: json.dump(data, f, ensure_ascii=False)
            with out: clear_output(); print(f"تم حفظ {key_in.value}")

    btn = widgets.Button(description="حفظ", button_style='success')
    btn.on_click(save)
    display(widgets.VBox([key_in, val_in, btn, out]))

# edit_correction_dict_ui() # قم بإلغاء التعليق للتشغيل

In [ ]:
def launch_review_ui_v2():
    import sqlite3, pandas as pd, ipywidgets as widgets, os
    from IPython.display import display
    from datetime import datetime

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY confidence ASC", conn)
    if df.empty:
        print("✅ لا توجد كلمات للمراجعة."); return

    current = 0
    img = widgets.Image(format='png', width=350)
    text = widgets.Text(description="النص:")
    conf_bar = widgets.FloatProgress(min=0, max=1.0, description='الثقة:', layout={'width': '95%'})
    conf_label = widgets.HTML(value="")
    prog = widgets.IntProgress(min=0, max=len(df)-1, bar_style='info')
    info = widgets.Label()

    def update():
        nonlocal current
        if 0 <= current < len(df):
            row = df.iloc[current]
            img.value = row['image_data']
            text.value = str(row['predicted_text'] or '')
            c = float(row['confidence'])
            conf_bar.value = c
            conf_bar.bar_style = 'danger' if c < 0.5 else ('warning' if c < 0.8 else 'success')
            conf_label.value = f"<b>{c:.2%}</b>"
            info.value = f"{current+1}/{len(df)} | صفحة {row['page_num']}"
            prog.value = current
        else:
            img.value = b''; text.value = ''; info.value = "🏁 اكتملت المراجعة"

    def on_confirm(b):
        nonlocal current, df
        if not (0 <= current < len(df)): return
        rid = int(df.iloc[current]['image_id'])
        original = df.iloc[current]['predicted_text']
        corrected = text.value

        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("UPDATE handwriting_data SET predicted_text=?, status='verified' WHERE image_id=?", (corrected, rid))

        if original != corrected:
            pd.DataFrame([{
                'timestamp': datetime.now().isoformat(),
                'image_id': rid,
                'original_text': original,
                'corrected_text': corrected,
                'status': 'verified'
            }]).to_csv(FEEDBACK_CSV, mode='a', header=not os.path.exists(FEEDBACK_CSV), index=False, encoding='utf-8')

        df = df.drop(df.index[current]).reset_index(drop=True)
        prog.max = max(0, len(df)-1)
        if current >= len(df) and len(df) > 0: current = len(df) - 1
        update()

    def on_next(b):
        nonlocal current
        current = min(len(df)-1, current + 1)
        update()

    def on_prev(b):
        nonlocal current
        current = max(0, current - 1)
        update()

    def on_delete(b):
        nonlocal current, df
        if not (0 <= current < len(df)): return
        rid = int(df.iloc[current]['image_id'])
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("DELETE FROM handwriting_data WHERE image_id=?", (rid,))
        df = df.drop(df.index[current]).reset_index(drop=True)
        prog.max = max(0, len(df)-1)
        if current >= len(df) and len(df) > 0: current = len(df)-1
        update()

    btn_confirm = widgets.Button(description="✅ تأكيد", button_style='success')
    btn_next = widgets.Button(description="التالي", button_style='info')
    btn_prev = widgets.Button(description="⬅️ السابق", button_style='info')
    btn_del = widgets.Button(description="🗑️ حذف", button_style='danger')

    btn_confirm.on_click(on_confirm)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_del.on_click(on_delete)

    display(widgets.VBox([
        prog, info, img, text,
        widgets.HBox([conf_bar, conf_label]),
        widgets.HBox([btn_prev, btn_confirm, btn_del, btn_next])
    ]))
    update()

# تشغيل واجهة مراجعة الكلمات
launch_review_ui_v2()

### **الخلية 9: واجهة مراجعة الجمل (Sentence-Level Review UI)**
تتيح هذه الواجهة مراجعة الكلمات مجمعة في جمل لتحسين دقة التصحيح بناءً على السياق، مع تحديث تلقائي لقاموس التصحيحات.

In [ ]:
def launch_sentence_review_ui(y_tolerance=25):
    import sqlite3, pandas as pd, ipywidgets as widgets, os, io as _io
    from IPython.display import display, clear_output
    from PIL import Image
    from datetime import datetime

    with sqlite3.connect(DB_PATH) as conn:
        words_df = pd.read_sql_query("SELECT * FROM handwriting_data ORDER BY page_num, y, x", conn)

    if words_df.empty: return print("⚠️ لا توجد بيانات للمراجعة.")

    sentences = []
    for page in words_df['page_num'].unique():
        p_words = words_df[words_df['page_num'] == page].copy()
        if p_words.empty: continue

        curr_line = [p_words.iloc[0].to_dict()]
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                curr_line.append(row)
            else:
                sentences.append(curr_line); curr_line = [row]
        sentences.append(curr_line)

    current_idx = 0

    img_container = widgets.HBox(layout={'overflow_x': 'scroll', 'padding': '10px'})
    sentence_input = widgets.Textarea(description="الجملة:", layout={'width': '95%', 'height': '80px'})
    info_label = widgets.Label()
    progress = widgets.IntProgress(min=0, max=len(sentences)-1, layout={'width': '95%', 'height': '20px'})

    def get_img_widget(blob):
        img = Image.open(_io.BytesIO(blob))
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        return widgets.Image(value=buf.getvalue(), format='png', width=120)

    def update_ui():
        nonlocal current_idx
        if not (0 <= current_idx < len(sentences)): return
        sent = sentences[current_idx]
        img_container.children = [get_img_widget(w['image_data']) for w in sent]
        original_text = " ".join([str(w['predicted_text'] or "") for w in sent])
        sentence_input.value = original_text
        info_label.value = f"جملة {current_idx+1} من {len(sentences)} | صفحة {sent[0]['page_num']}"
        progress.value = current_idx

    def save_current(b):
        nonlocal current_idx
        sent = sentences[current_idx]
        original = " ".join([str(w['predicted_text'] or "") for w in sent])
        corrected = sentence_input.value.strip()
        if not corrected: return

        with sqlite3.connect(DB_PATH) as conn:
            for w in sent:
                conn.execute("UPDATE handwriting_data SET status='sentence_corrected' WHERE image_id=?", (w['image_id'],))

        sent_id = f"p{sent[0]['page_num']}_y{sent[0]['y']}"
        if original != corrected:
            pd.DataFrame([{
                'timestamp': datetime.now().isoformat(),
                'image_id': None,
                'original_text': original,
                'corrected_text': corrected,
                'status': f'sent_rev_{sent_id}'
            }]).to_csv(FEEDBACK_CSV, mode='a', header=not os.path.exists(FEEDBACK_CSV), index=False, encoding='utf-8')

            orig_words = original.split()
            corr_words = corrected.split()
            if len(orig_words) == len(corr_words):
                derived = []
                for o, c in zip(orig_words, corr_words):
                    if o != c:
                        derived.append({'timestamp': datetime.now().isoformat(), 'image_id': None, 'original_text': o, 'corrected_text': c, 'status': 'sentence_derived'})
                if derived: pd.DataFrame(derived).to_csv(FEEDBACK_CSV, mode='a', header=False, index=False, encoding='utf-8')
        print(f"✅ تم حفظ الجملة {current_idx+1}")
        current_idx = min(len(sentences)-1, current_idx + 1)
        update_ui()

    def on_next(b):
        nonlocal current_idx
        current_idx = min(len(sentences)-1, current_idx + 1)
        update_ui()

    def on_prev(b):
        nonlocal current_idx
        current_idx = max(0, current_idx - 1)
        update_ui()

    btn_save = widgets.Button(description="✅ حفظ وتأكيد", button_style='success')
    btn_next = widgets.Button(description="التالي ⮕", button_style='info')
    btn_prev = widgets.Button(description="⬅ السابق", button_style='info')
    btn_save.on_click(save_current)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)

    display(widgets.VBox([progress, info_label, img_container, sentence_input, widgets.HBox([btn_prev, btn_save, btn_next])]))
    update_ui()

launch_sentence_review_ui()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# التأكد من وجود قاعدة البيانات والجدول قبل القراءة
with sqlite3.connect(DB_PATH) as conn:
    conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
                    (image_id INTEGER PRIMARY KEY AUTOINCREMENT,
                     image_data BLOB, predicted_text TEXT, status TEXT,
                     confidence REAL, model_source TEXT,
                     x INTEGER, y INTEGER, w INTEGER, h INTEGER,
                     page_num INTEGER, cached_predictions TEXT)''')

try:
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("SELECT * FROM handwriting_data ORDER BY image_id", conn)
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.DataFrame()

current_idx = 0
img_disp = widgets.Image(format='png', width=500)
text_input = widgets.Text(description="التصحيح:", layout={'width': '95%'})
info_lbl = widgets.Label()
btns = [widgets.Button(layout={'width': '32%'}, button_style='warning') for _ in range(3)]

def update_ui():
    global current_idx
    if not df.empty and current_idx < len(df):
        row = df.iloc[current_idx]
        img_disp.value = row['image_data']
        if row.get('cached_predictions') and pd.notna(row['cached_predictions']):
            preds = json.loads(row['cached_predictions'])
        else:
            preds = get_ensemble_predictions(row['image_data'])
            with sqlite3.connect(DB_PATH) as c:
                c.execute("UPDATE handwriting_data SET cached_predictions=? WHERE image_id=?", (json.dumps(preds), int(row['image_id'])))
        for i in range(3):
            if i < len(preds):
                btns[i].description = preds[i][:60]
                btns[i].disabled = False
            else:
                btns[i].disabled = True
        text_input.value = preds[0] if preds else ""
        info_lbl.value = f"ID: {row['image_id']} | {current_idx+1}/{len(df)} | المصادق: {df[df['status']=='verified'].shape[0]}"
    else:
        info_lbl.value = "⚠️ لا توجد بيانات للمراجعة أو اكتملت المراجعة!"

def on_next(_):
    global current_idx
    if not df.empty and current_idx < len(df):
        val = text_input.value.strip()
        img_id = int(df.iloc[current_idx]['image_id'])
        with sqlite3.connect(DB_PATH) as c:
            c.execute("UPDATE handwriting_data SET predicted_text=?, status='verified' WHERE image_id=?", (val, img_id))
        current_idx += 1
        update_ui()

def on_btn_click(btn):
    text_input.value = btn.description
    on_next(btn)

for b in btns:
    b.on_click(on_btn_click)

next_btn = widgets.Button(description="التالي ➡️", button_style='success')
next_btn.on_click(on_next)
vocab_btn = widgets.Button(description="استخراج القاموس 📖", button_style='info')
hf_btn = widgets.Button(description="رفع لـ Hugging Face 🤗", button_style='primary')
vocab_btn.on_click(lambda x: display(extract_bilingual_vocab()))
hf_btn.on_click(lambda x: upload_to_hf())

display(widgets.VBox([
    widgets.HTML("<h2>🩺 منظومة الدكتور عبد الملك للأرشفة والذكاء الاصطناعي</h2>"),
    info_lbl, img_disp, widgets.HBox(btns), text_input,
    widgets.HBox([next_btn, vocab_btn, hf_btn])
]))
update_ui()